# 365 Probabilidades - Dia #126
## Qual a probabilidade de as crianças de hoje se machucarem menos do que as de antes?

**Tipo:** Tendência
**Data de publicação:** 2026-10-17
**Ferramenta:** Python
**Decisão analisada:** A ideia de que criança de hoje se quebra menos aparece no registro?
**Hashtag:** #365Probabilidades #Dia126

---

### 📖 A História

É comum ouvir que criança de antigamente se machucava mais. Subia em árvore, andava de
bicicleta sem capacete, brincava na rua até escurecer. A de hoje ficaria mais em casa e
mais na tela, e por isso se quebraria menos.

Fui ver o que o registro diz.

---

### 📚 O Conceito: contar não basta, é preciso dividir

O Brasil não tem um registro de todas as fraturas de crianças. O que existe é o registro
das internações pagas pelo SUS, com o diagnóstico, a idade e o sexo. Uso as internações
por fratura como medida do machucado sério, aquele que levou a criança ao hospital.

Há um cuidado antes de comparar anos. O número de crianças no país diminuiu. Em 2008
havia 49,2 milhões de pessoas de 0 a 14 anos. Em 2025 são 41,6 milhões. Com menos
crianças, o mesmo número de internações significa mais internações por criança. Por isso
a medida deste notebook é a **taxa por 100 mil crianças**, e não a contagem.

Há um segundo cuidado. O registro mudou três vezes, e cada mudança quebra a série. A
tendência é medida dentro de cada trecho comparável, e nunca de um trecho para o outro.

---

### 🧮 O Modelo

Seis camadas. As contagens vêm do DataSUS, a população vem do IBGE e os planos de saúde
vêm da ANS. As contas são nossas.

1. **Contagem e taxa.** As internações por fratura de 0 a 14 anos, de 2008 a 2025, em
   número e por 100 mil crianças.
2. **A tendência em cada trecho.** Regressão log-linear da taxa sobre o ano, em quatro
   trechos. Sai a variação anual com intervalo de 95% e a probabilidade de a tendência
   ser de queda.
3. **Por sexo e idade.** Seis grupos, de 2008 a 2025. Quantos caíram.
4. **É a fratura ou é o sistema?** A fratura comparada com todas as outras internações de
   crianças. Se as duas sobem juntas, a alta é do SUS internando mais.
5. **SUS e rede particular.** A parte das crianças com plano de saúde em cada ano, e a
   taxa recalculada só sobre as crianças sem plano.
6. **A chance de uma criança.** A taxa anual vira a chance de uma internação por fratura
   antes dos 15 anos.

**Dados:**
- Ministério da Saúde, DataSUS, *Sistema de Informações Hospitalares do SUS (SIH/SUS)*,
  Morbidade Hospitalar. Internações com diagnóstico de fratura, por faixa etária e sexo.
  Consulta no TabNet em 11/10/2026. Três trechos: **1992 a 1997** (CID-9, categoria
  "Fraturas" da lista básica, por local de internação), **1998 a 2007** (CID-10, cinco
  categorias de fratura da lista de morbidade, por local de internação) e **2008 a 2025**
  (CID-10, as mesmas cinco categorias, por local de residência e ano de atendimento).
  De 1984 a 1991 o registro existe, mas sem idade.
- IBGE, *Projeções da População do Brasil e Unidades da Federação por sexo e idade:
  2000-2070* (edição 2024), para 2000 a 2025. Para 1992 a 1999, as estimativas
  populacionais antigas do IBGE publicadas no TabNet.
- ANS, *Sala de Situação / TabNet*, beneficiários de planos de assistência médica por
  faixa etária, em dezembro de cada ano, de 2008 a 2025. Consulta em 11/10/2026.
- Os resumos estão em `dia-126-serie-longa.csv` e `dia-126-fraturas-sexo-idade.csv`, ao
  lado deste notebook.

**Fonte de apoio:**
- Coorte de nascimentos de Pelotas de 1993. "The role of early life variables on the risk
  of fractures from birth to early adolescence: a prospective birth cohort study".
  *Osteoporosis International* (2009). DOI 10.1007/s00198-009-0889-y. 4.452 crianças
  acompanhadas até os 11 anos (87,5% da coorte). **14,2%** tiveram ao menos uma fratura
  (IC 95% de 13,2% a 15,2%), segundo o relato da mãe. Lido no texto do artigo.
- Lempesis, V., Rosengren, B. E., Nilsson, J.-Å., Landin, L., Tiderius, C. J. & Karlsson,
  M. K. (2017). "Time trends in pediatric fracture incidence in Sweden during the period
  1950-2006". *Acta Orthopaedica* 88(4), 440-445. Em Malmö, a taxa ajustada por idade não
  mudou entre 1993-94 e 2005-06 (razão 0,9, IC 95% de 0,8 a 1,03).

**Nota metodológica.** O fator ×0,80 do projeto não se aplica. Ele vale para proporções
de pesquisa de opinião autorrelatada. Aqui o dado é registro administrativo de
internação.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = 'Fonte: DataSUS, SIH/SUS · IBGE, Projeções da População (ed. 2024) · cálculo próprio | #365Probabilidades'

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def sinal(x, casas=1):
    return ("+" if x >= 0 else "−") + vir(abs(x), casas)

def abre(arquivo):
    for pasta in ["", os.path.join("modelos", "dias-121-150"), os.path.join("..", "dias-121-150")]:
        caminho = os.path.join(pasta, arquivo)
        if os.path.exists(caminho):
            return pd.read_csv(caminho)
    raise FileNotFoundError(arquivo + " precisa estar na mesma pasta do notebook")

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS ---
serie = abre("dia-126-serie-longa.csv")
painel = abre("dia-126-fraturas-sexo-idade.csv")

serie['taxa'] = serie['fraturas'] / serie['pop_0a14'] * 1e5
nova = serie[serie['ano'] >= 2008].set_index('ano')

# Coorte de Pelotas de 1993 (apoio)
PELOTAS_N, PELOTAS_FRATURA = 4423, 628            # crianças com dado completo · com ao menos uma fratura
PELOTAS_P = PELOTAS_FRATURA / PELOTAS_N

print("=" * 72)
print("  DADOS · SIH/SUS E IBGE")
print("=" * 72)
print(f"\n  Série: {serie['ano'].min()} a {serie['ano'].max()} · {len(serie)} anos · três trechos")
for trecho, g in serie.groupby('trecho', sort=False):
    print(f"    {trecho}: {mil(g['fraturas'].sum())} internações por fratura de 0 a 14 anos")
print(f"\n  Crianças de 0 a 14 anos no Brasil:")
for ano in [2008, 2019, 2025]:
    print(f"    {ano}: {vir(nova.loc[ano, 'pop_0a14'] / 1e6)} milhões")
print(f"    variação 2008-2025: {sinal((nova.loc[2025, 'pop_0a14'] / nova.loc[2008, 'pop_0a14'] - 1) * 100)}%")
print(f"\n  Pelotas, coorte de 1993: {PELOTAS_FRATURA} de {mil(PELOTAS_N)} crianças com fratura até os 11 anos "
      f"({vir(PELOTAS_P * 100)}%)")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# CAMADA 1 · Contagem e taxa
print("=" * 72)
print("  CAMADA 1 · CONTAGEM E TAXA (0 a 14 anos, 2008 a 2025)")
print("=" * 72)
print(f"\n  {'ano':>6} {'internações':>13} {'por 100 mil':>13}")
for ano in [2008, 2012, 2016, 2019, 2020, 2022, 2023, 2024, 2025]:
    print(f"  {ano:>6} {mil(nova.loc[ano, 'fraturas']):>13} {vir(nova.loc[ano, 'taxa']):>13}")
var_cont = (nova.loc[2025, 'fraturas'] / nova.loc[2008, 'fraturas'] - 1) * 100
var_taxa = (nova.loc[2025, 'taxa'] / nova.loc[2008, 'taxa'] - 1) * 100
var_cont19 = (nova.loc[2019, 'fraturas'] / nova.loc[2008, 'fraturas'] - 1) * 100
var_taxa19 = (nova.loc[2019, 'taxa'] / nova.loc[2008, 'taxa'] - 1) * 100
print(f"\n  2008 a 2019: contagem {sinal(var_cont19)}% · taxa {sinal(var_taxa19)}%")
print(f"  2008 a 2025: contagem {sinal(var_cont)}% · taxa {sinal(var_taxa)}%")
print("  A contagem até 2019 cai. A taxa sobe, porque há menos crianças.")

# CAMADA 2 · A tendência em cada trecho
def tendencia(anos, taxas):
    # regressão log-linear: ln(taxa) = a + b·ano. A variação anual é e^b − 1.
    x = np.asarray(anos, float)
    y = np.log(np.asarray(taxas, float))
    r = stats.linregress(x, y)
    gl = len(x) - 2
    tc = stats.t.ppf(0.975, gl)
    anual = (np.exp(r.slope) - 1) * 100
    lo = (np.exp(r.slope - tc * r.stderr) - 1) * 100
    hi = (np.exp(r.slope + tc * r.stderr) - 1) * 100
    p_queda = stats.t.cdf(-r.slope / r.stderr, gl)        # P(inclinação < 0)
    return dict(anual=anual, lo=lo, hi=hi, p_queda=p_queda, n=len(x), r=r)

def pct_pequeno(p):
    return "menos de 0,1%" if p < 0.001 else vir(p * 100, 1) + "%"

TRECHOS = {
    '1992 a 1997': (1992, 1997),
    '1998 a 2007': (1998, 2007),
    '2008 a 2019': (2008, 2019),
    '2008 a 2025': (2008, 2025),
}
print("\n" + "=" * 72)
print("  CAMADA 2 · A TENDÊNCIA EM CADA TRECHO COMPARÁVEL")
print("=" * 72)
print(f"\n  {'trecho':<14} {'anos':>5} {'variação por ano':>18} {'IC 95%':>18} {'P(queda)':>14}")
tend = {}
for nome, (a, b) in TRECHOS.items():
    g = serie[(serie['ano'] >= a) & (serie['ano'] <= b)]
    t = tendencia(g['ano'], g['taxa'])
    tend[nome] = t
    print(f"  {nome:<14} {t['n']:>5} {sinal(t['anual'], 2) + '%':>18} "
          f"{'[' + sinal(t['lo'], 2) + '; ' + sinal(t['hi'], 2) + ']':>18} {pct_pequeno(t['p_queda']):>14}")
P_QUEDA = tend['2008 a 2019']['p_queda']
print(f"\n  No trecho mais recente sem pandemia (2008 a 2019), a taxa subiu "
      f"{vir(tend['2008 a 2019']['anual'], 2)}% ao ano.")
print(f"  A probabilidade de a tendência nesse trecho ser de queda é de {pct_pequeno(P_QUEDA)}.")
print("  A regressão trata os anos como independentes. Com 12 pontos, o intervalo deve ser lido com folga.")
print("  O único trecho com queda é 1992 a 1997. Ele termina na troca de classificação de 1998.")

# CAMADA 3 · Por sexo e idade
painel['taxa_masc'] = painel['masc'] / painel['pop_masc'] * 1e5
painel['taxa_fem'] = painel['fem'] / painel['pop_fem'] * 1e5
ROTULO = {'0a4': '0 a 4 anos', '5a9': '5 a 9 anos', '10a14': '10 a 14 anos'}
print("\n" + "=" * 72)
print("  CAMADA 3 · POR SEXO E IDADE (taxa por 100 mil)")
print("=" * 72)
print(f"\n  {'grupo':<24} {'2008':>7} {'2019':>7} {'2025':>7} {'2008-19':>9} {'2008-25':>9} {'ao ano até 2019':>17}")
grupos = {}
for faixa in ['0a4', '5a9', '10a14']:
    g = painel[painel['faixa'] == faixa].set_index('ano')
    for sexo, col in [('meninos', 'taxa_masc'), ('meninas', 'taxa_fem')]:
        t = tendencia(g.loc[2008:2019].index, g.loc[2008:2019, col])
        v19 = (g.loc[2019, col] / g.loc[2008, col] - 1) * 100
        v25 = (g.loc[2025, col] / g.loc[2008, col] - 1) * 100
        grupos[(faixa, sexo)] = dict(serie=g[col], v19=v19, v25=v25, t=t)
        print(f"  {sexo + ', ' + ROTULO[faixa]:<24} {vir(g.loc[2008, col], 0):>7} {vir(g.loc[2019, col], 0):>7} "
              f"{vir(g.loc[2025, col], 0):>7} {sinal(v19) + '%':>9} {sinal(v25) + '%':>9} "
              f"{sinal(t['anual'], 2) + '%':>17}")
cairam19 = sum(1 for v in grupos.values() if v['v19'] < 0)
cairam25 = sum(1 for v in grupos.values() if v['v25'] < 0)
print(f"\n  Grupos com taxa menor em 2019 do que em 2008: {cairam19} de 6")
print(f"  Grupos com taxa menor em 2025 do que em 2008: {cairam25} de 6")
tot = painel.groupby('ano')[['masc', 'fem']].sum()
print(f"  Meninos internados para cada menina: {vir(tot.loc[2008, 'masc'] / tot.loc[2008, 'fem'])} em 2008 · "
      f"{vir(tot.loc[2025, 'masc'] / tot.loc[2025, 'fem'])} em 2025")

# CAMADA 4 · É a fratura ou é o sistema?
nova = nova.copy()
nova['outras'] = nova['internacoes_todas'] - nova['fraturas']
nova['taxa_outras'] = nova['outras'] / nova['pop_0a14'] * 1e5
nova['fatia'] = nova['fraturas'] / nova['internacoes_todas'] * 100
print("\n" + "=" * 72)
print("  CAMADA 4 · É A FRATURA OU É O SISTEMA?")
print("=" * 72)
print(f"\n  {'ano':>6} {'fratura/100 mil':>16} {'outras causas/100 mil':>23} {'fatia das fraturas':>20}")
for ano in [2008, 2012, 2016, 2019, 2022, 2023, 2024, 2025]:
    print(f"  {ano:>6} {vir(nova.loc[ano, 'taxa']):>16} {vir(nova.loc[ano, 'taxa_outras'], 0):>23} "
          f"{vir(nova.loc[ano, 'fatia'], 2) + '%':>20}")
alta_fr = (nova.loc[2025, 'taxa'] / nova.loc[2019, 'taxa'] - 1) * 100
alta_out = (nova.loc[2025, 'taxa_outras'] / nova.loc[2019, 'taxa_outras'] - 1) * 100
print(f"\n  De 2019 a 2025: fraturas {sinal(alta_fr)}% · todas as outras causas {sinal(alta_out)}%")
print(f"  As duas sobem juntas. A fatia das fraturas vai de {vir(nova.loc[2019, 'fatia'], 2)}% para "
      f"{vir(nova.loc[2025, 'fatia'], 2)}%.")
print(f"  A alta depois de 2022 é, na maior parte, o SUS internando mais crianças por qualquer causa.")
alta_fr08 = (nova.loc[2019, 'taxa'] / nova.loc[2008, 'taxa'] - 1) * 100
alta_out08 = (nova.loc[2019, 'taxa_outras'] / nova.loc[2008, 'taxa_outras'] - 1) * 100
print(f"  De 2008 a 2019: fraturas {sinal(alta_fr08)}% · outras causas {sinal(alta_out08)}%. "
      f"As outras causas ficaram paradas, e a fratura subiu.")

# CAMADA 5 · SUS e rede particular
nova['cobertura'] = nova['benef_plano_0a14'] / nova['pop_0a14'] * 100
nova['sem_plano'] = nova['pop_0a14'] - nova['benef_plano_0a14']
nova['taxa_sem_plano'] = nova['fraturas'] / nova['sem_plano'] * 1e5
print("\n" + "=" * 72)
print("  CAMADA 5 · SUS E REDE PARTICULAR")
print("=" * 72)
print(f"\n  {'ano':>6} {'com plano':>12} {'cobertura':>11} {'sem plano':>12} {'taxa geral':>12} {'taxa sem plano':>16}")
for ano in [2008, 2011, 2014, 2017, 2019, 2022, 2025]:
    print(f"  {ano:>6} {vir(nova.loc[ano, 'benef_plano_0a14'] / 1e6) + ' mi':>12} "
          f"{vir(nova.loc[ano, 'cobertura']) + '%':>11} {vir(nova.loc[ano, 'sem_plano'] / 1e6) + ' mi':>12} "
          f"{vir(nova.loc[ano, 'taxa']):>12} {vir(nova.loc[ano, 'taxa_sem_plano']):>16}")
var_sem = (nova.loc[2025, 'sem_plano'] / nova.loc[2008, 'sem_plano'] - 1) * 100
print(f"\n  A parte das crianças com plano foi de {vir(nova.loc[2008, 'cobertura'])}% para "
      f"{vir(nova.loc[2025, 'cobertura'])}%.")
print(f"  As crianças sem plano passaram de {vir(nova.loc[2008, 'sem_plano'] / 1e6)} para "
      f"{vir(nova.loc[2025, 'sem_plano'] / 1e6)} milhões ({sinal(var_sem)}%).")
for a, b_ in [(2008, 2019), (2008, 2025)]:
    v1 = (nova.loc[b_, 'taxa'] / nova.loc[a, 'taxa'] - 1) * 100
    v2 = (nova.loc[b_, 'taxa_sem_plano'] / nova.loc[a, 'taxa_sem_plano'] - 1) * 100
    print(f"  {a} a {b_}: taxa geral {sinal(v1)}% · taxa por criança sem plano {sinal(v2)}%")
t_sem = tendencia(nova.loc[2008:2019].index, nova.loc[2008:2019, 'taxa_sem_plano'])
print(f"  Tendência 2008 a 2019 da taxa sem plano: {sinal(t_sem['anual'], 2)}% ao ano "
      f"[{sinal(t_sem['lo'], 2)}; {sinal(t_sem['hi'], 2)}]")
print("  A migração entre plano e SUS não explica a alta. Com ela na conta, a alta fica maior.")
print("  Premissa: nenhuma criança com plano é internada pelo SUS. Na prática algumas são,")
print("  então a taxa por criança sem plano é um limite superior.")

# CAMADA 6 · A chance de uma criança
print("\n" + "=" * 72)
print("  CAMADA 6 · A CHANCE DE UMA CRIANÇA")
print("=" * 72)
um_em = 1e5 / nova.loc[2025, 'taxa']
print(f"\n  Em 2025: {vir(nova.loc[2025, 'taxa'], 0)} internações por 100 mil crianças. "
      f"Uma em cada {mil(um_em)} crianças por ano.")
acum = {}
for sexo, col in [('meninos', 'taxa_masc'), ('meninas', 'taxa_fem')]:
    for ano in [2008, 2025]:
        soma = sum(painel[(painel['faixa'] == f) & (painel['ano'] == ano)][col].iloc[0] * 5 for f in ROTULO) / 1e5
        acum[(sexo, ano)] = 1 - np.exp(-soma)
    print(f"  Chance de ao menos uma internação por fratura antes dos 15 anos, {sexo}: "
          f"{vir(acum[(sexo, 2008)] * 100)}% com as taxas de 2008 · {vir(acum[(sexo, 2025)] * 100)}% com as de 2025")
print("  Premissa: a criança vive os 15 anos com as taxas de um mesmo ano. Reinternações contam como novas.")
ic_pel = stats.beta(PELOTAS_FRATURA + 0.5, PELOTAS_N - PELOTAS_FRATURA + 0.5).interval(0.95)
print(f"\n  Referência · Pelotas, coorte de 1993: {vir(PELOTAS_P * 100)}% das crianças com ao menos uma fratura até os 11 anos")
print(f"  IC 95% recalculado [{vir(ic_pel[0] * 100)}; {vir(ic_pel[1] * 100)}] · uma em cada {vir(1 / PELOTAS_P, 0)}")
print("  A internação é só a parte mais grave. A maioria das fraturas é tratada sem internar.")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---
CORES_TRECHO = {'1992-1997 (CID-9)': CINZA, '1998-2007 (CID-10)': DOURADO, '2008-2025 (tabela nova)': VERMELHO}

# GRÁFICO 1 · A série de 34 anos
fig1, ax1 = plt.subplots(figsize=(12, 8))
for trecho, g in serie.groupby('trecho', sort=False):
    ax1.plot(g['ano'], g['taxa'], color=CORES_TRECHO[trecho], linewidth=3, marker='o', markersize=5, label=trecho)
for x, txt in [(1997.5, 'muda a\nclassificação'), (2007.5, 'muda a tabela\ndo SUS')]:
    ax1.axvline(x, color=CINZA, linestyle=':', linewidth=1.5)
    ax1.text(x + 0.2, 104, txt, fontsize=10, color=CINZA, va='bottom')
ax1.axvspan(2019.5, 2021.5, color=CINZA, alpha=0.12)
ax1.text(2020.5, 104, 'pandemia', fontsize=10, color=CINZA, ha='center', va='bottom')
for ano in [1992, 2008, 2025]:
    v = serie.set_index('ano').loc[ano, 'taxa']
    ax1.annotate(vir(v, 0), (ano, v), xytext=(0, 10), textcoords='offset points', ha='center', fontsize=12,
                 fontweight='bold')
ax1.set_ylim(100, 215)
ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Internações por fratura por 100 mil crianças de 0 a 14 anos', fontsize=12)
ax1.set_title('Em 34 anos de registro, a taxa não diminuiu\n'
              'cada cor é um trecho comparável', fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='upper left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-126-grafico-01-a-serie.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · Contagem, população e taxa
fig2, ax2 = plt.subplots(figsize=(12, 8))
base = nova.loc[2008]
ax2.plot(nova.index, nova['fraturas'] / base['fraturas'] * 100, color=CINZA, linewidth=2.5, linestyle='--',
         label='internações por fratura (contagem)')
ax2.plot(nova.index, nova['pop_0a14'] / base['pop_0a14'] * 100, color=DOURADO, linewidth=2.5,
         label='crianças de 0 a 14 anos')
ax2.plot(nova.index, nova['taxa'] / base['taxa'] * 100, color=VERMELHO, linewidth=3.5,
         label='taxa por 100 mil crianças')
ax2.axhline(100, color='#333', linewidth=1)
for col, cor, dy in [('taxa', VERMELHO, 8), ('pop_0a14', DOURADO, -16)]:
    v = nova.loc[2025, col] / base[col] * 100
    ax2.annotate(sinal(v - 100, 0) + '%', (2025, v), xytext=(0, dy), textcoords='offset points', ha='center',
                 color=cor, fontsize=13, fontweight='bold')
ax2.set_xticks(range(2008, 2026, 2))
ax2.set_xlabel('Ano', fontsize=12)
ax2.set_ylabel('Índice (2008 = 100)', fontsize=12)
ax2.set_title('Menos crianças, e a taxa de internação por fratura maior\n'
              'três séries com 2008 = 100', fontsize=14, pad=16)
ax2.legend(frameon=False, fontsize=11, loc='upper left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-126-grafico-02-contagem-e-taxa.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Por sexo e idade
fig3, eixos = plt.subplots(1, 3, figsize=(14, 7), sharey=True)
for ax, faixa in zip(eixos, ['0a4', '5a9', '10a14']):
    for sexo, cor in [('meninos', VERDE), ('meninas', VERMELHO)]:
        s = grupos[(faixa, sexo)]['serie']
        ax.plot(s.index, s.values, color=cor, linewidth=3, label=sexo)
        ax.annotate(f"{vir(s.loc[2025], 0)}\n({sinal(grupos[(faixa, sexo)]['v25'], 0)}%)", (2025, s.loc[2025]),
                    xytext=(-4, -30 if (faixa == '0a4' and sexo == 'meninas') else 8), textcoords='offset points',
                    ha='right', fontsize=11, color=cor)
    ax.set_title(ROTULO[faixa], fontsize=13)
    ax.set_xticks([2008, 2014, 2020, 2025])
    ax.set_ylim(0, 450)
eixos[0].set_ylabel('Internações por fratura por 100 mil', fontsize=12)
eixos[0].legend(frameon=False, fontsize=12, loc='upper left')
fig3.suptitle('Nenhum grupo tem taxa menor em 2025 do que em 2008\n'
              'entre parênteses, a variação de 2008 a 2025', fontsize=14)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 0.98))
plt.savefig('dia-126-grafico-03-sexo-e-idade.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · É a fratura ou é o sistema?
fig4, ax4 = plt.subplots(figsize=(12, 8))
ax4.plot(nova.index, nova['taxa'] / nova.loc[2008, 'taxa'] * 100, color=VERMELHO, linewidth=3.5,
         label='internações por fratura')
ax4.plot(nova.index, nova['taxa_outras'] / nova.loc[2008, 'taxa_outras'] * 100, color=VERDE, linewidth=3.5,
         label='internações por todas as outras causas')
ax4.axhline(100, color='#333', linewidth=1)
ax4.axvspan(2019.5, 2021.5, color=CINZA, alpha=0.12)
ax4.text(2020.5, 66, 'pandemia', fontsize=10, color=CINZA, ha='center')
ax4.set_xticks(range(2008, 2026, 2))
ax4.set_ylim(60, 135)
ax4.set_xlabel('Ano', fontsize=12)
ax4.set_ylabel('Taxa por 100 mil crianças (2008 = 100)', fontsize=12)
ax4.set_title('Depois de 2022, fraturas e outras causas sobem juntas\n'
              'internações de crianças de 0 a 14 anos no SUS', fontsize=14, pad=16)
ax4.legend(frameon=False, fontsize=12, loc='upper left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-126-grafico-04-fratura-ou-sistema.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# GRÁFICO 5 · SUS e rede particular
fig5, (ax5, ax6) = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [1, 1.3]})
ax5.plot(nova.index, nova['cobertura'], color=DOURADO, linewidth=3.5)
for ano in [2008, 2014, 2025]:
    ax5.annotate(vir(nova.loc[ano, 'cobertura']) + '%', (ano, nova.loc[ano, 'cobertura']),
                 xytext=(34, -6) if ano == 2008 else (0, 10),
                 textcoords='offset points', ha='center', fontsize=12, fontweight='bold')
ax5.set_ylim(14, 26)
ax5.set_xticks([2008, 2012, 2016, 2020, 2025])
ax5.set_ylabel('Crianças de 0 a 14 anos com plano de saúde (%)', fontsize=12)
ax5.set_title('Mais crianças com plano', fontsize=13)
ax6.plot(nova.index, nova['taxa'], color=VERMELHO, linewidth=3.5, label='por 100 mil crianças')
ax6.plot(nova.index, nova['taxa_sem_plano'], color=VERDE, linewidth=3.5, label='por 100 mil crianças sem plano')
for col, cor in [('taxa', VERMELHO), ('taxa_sem_plano', VERDE)]:
    for ano, dx in [(2008, 0), (2025, -6)]:
        ax6.annotate(vir(nova.loc[ano, col], 0), (ano, nova.loc[ano, col]), xytext=(dx, 10),
                     textcoords='offset points', ha='center', fontsize=12, fontweight='bold', color=cor)
ax6.set_ylim(120, 285)
ax6.set_xticks([2008, 2012, 2016, 2020, 2025])
ax6.set_ylabel('Internações por fratura no SUS', fontsize=12)
ax6.set_title('A taxa calculada de dois jeitos', fontsize=13)
ax6.legend(frameon=False, fontsize=11, loc='upper left')
fig5.suptitle('A passagem de crianças para os planos não explica a alta no SUS', fontsize=14)
plt.figtext(0.5, 0.01, 'Fonte: DataSUS, SIH/SUS · ANS, beneficiários de planos · IBGE · cálculo próprio | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 0.96))
plt.savefig('dia-126-grafico-05-sus-e-planos.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**A probabilidade de a taxa de internação de crianças por fratura ter caído entre 2008 e
2019 é de menos de 0,1%.** No registro disponível, as crianças brasileiras não estão
sendo internadas menos por fratura.

O que os dados dizem, em ordem.

Entre 2008 e 2019, antes da pandemia, a taxa subiu 0,8% ao ano, com intervalo de 95% de
0,5% a 1,1%. Em contagem houve menos internações em 2019 do que em 2008, mas o país
tinha 5,1 milhões de crianças a menos. Por 100 mil crianças, a taxa foi de 153 para 164.

Os seis grupos de sexo e idade têm taxa maior em 2025 do que em 2008. O que menos mudou
foi o dos meninos de 5 a 9 anos, parado até 2019.

Na série de 34 anos, o único trecho de queda é o de 1992 a 1997. Ele termina numa troca
de classificação, e depois dela a taxa voltou a subir.

De 2022 em diante houve um salto. As internações de crianças por todas as outras causas
subiram junto, 16,5% contra 21,3% das fraturas. A maior parte desse salto acompanha o
SUS internando mais crianças por qualquer motivo.

A passagem de crianças entre plano de saúde e SUS não explica a alta. A parte das
crianças com plano subiu de 17% para 23,5%, e o número de crianças sem plano caiu 22%.
Calculada só sobre as crianças sem plano, a taxa sobe mais, de 185 para 260 por 100 mil.

Em 2025, uma em cada 504 crianças foi internada por fratura. Com as taxas de hoje, a
chance de passar por isso antes dos 15 anos é de 4% para um menino e de 1,7% para uma
menina. A internação é a parte mais grave. Em Pelotas, uma em cada sete crianças
nascidas em 1993 tinha quebrado algum osso até os 11 anos.

O registro mede internação, e não todo machucado. Ele não diz se as crianças caem mais
ou menos do que antes. Diz que a fratura que leva ao hospital continua acontecendo na
mesma frequência, ou em frequência maior.

A pergunta que fica:

*Como era a sua infância, e como você acha que ela apareceria num registro como este?*

---

### ⚠️ Limitações do Modelo

- **Internação não é fratura.** O registro só conta a fratura que precisou de hospital.
  Se o tratamento mudou e mais fraturas passaram a ser operadas, a internação sobe sem
  que as crianças se machuquem mais. O dado não separa uma coisa da outra.
- **Fratura não é todo machucado.** Cortes, torções e batidas sem fratura ficam de fora.
- **Só o SUS.** Internações pagas por plano de saúde ou pelo paciente não estão no
  registro. Não foi localizada uma série de internações por fratura na rede particular,
  por diagnóstico e idade. O notebook sabe o que aconteceu no SUS e não sabe o que
  aconteceu do lado dos planos.
- **A taxa por criança sem plano é um limite superior.** Ela supõe que nenhuma criança
  com plano é internada pelo SUS, e algumas são, principalmente em emergência.
- **A série tem três quebras.** A classificação mudou em 1998 e a tabela do SUS em 2008.
  Por isso a tendência é medida dentro de cada trecho, e nunca entre trechos.
- **O registro não volta à infância de quem tem mais de 40 anos.** Antes de 1992 não há
  idade no dado. A comparação é com os anos 1990 e 2000.
- A população de 1992 a 1999 vem das estimativas antigas do IBGE, que têm degraus (1996 é
  ano de contagem). As taxas desse trecho são menos firmes.
- A regressão trata cada ano como independente do anterior. Com 12 pontos, o intervalo é
  mais estreito do que deveria, e a probabilidade de queda deve ser lida como "muito
  baixa", sem a precisão das casas decimais.
- O salto de 2023 a 2025 coincide com a alta geral de internações de crianças. O notebook
  mostra as duas juntas e não identifica a causa.
- A chance até os 15 anos supõe que a criança vive todos os anos com as taxas de um mesmo
  ano, e conta reinternações como eventos novos.
- A coorte de Pelotas mede fratura pelo relato da mãe, numa cidade e numa geração. Entra
  como referência do tamanho do fenômeno, e não como tendência.
- Os dados foram consultados no TabNet em 11/10/2026. Os meses mais recentes de 2025
  ainda podem receber registros atrasados.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*